In [0]:
%sql
-- =====================================================================
-- BLOCK ONE — INGEST
-- Confirm the table names with:
SHOW TABLES IN workspace.default;
-- =====================================================================

database,tableName,isTemporary
default,inspection_violations_clean,false
default,rat_sightings,false
default,restaurant_inspections,false
default,rodent_complaints_clean,false
default,zip_service_gap,false


**Rat Sightings Data**

In [0]:
%sql
-- --- rat_sightings: row count -----------------------------------------
SELECT COUNT(*) AS row_count FROM workspace.default.rat_sightings;


row_count
50954


In [0]:
%sql
-- --- rat_sightings: date range ------------------------------------------
SELECT
    MIN(created_date) AS earliest_created,
    MAX(created_date) AS latest_created,
    MIN(closed_date)  AS earliest_closed,
    MAX(closed_date)  AS latest_closed
FROM workspace.default.rat_sightings;


earliest_created,latest_created,earliest_closed,latest_closed
2025-01-01T00:43:43.000Z,2026-09-17T01:35:48.000Z,2025-01-01T01:26:59.000Z,2026-09-16T19:03:00.000Z


In [0]:
%sql
-- rat_sightings: which columns are unusable 

-- constant column check — if min = max, the column carries zero information
SELECT
    COUNT(DISTINCT complaint_type) AS distinct_complaint_types,  -- 1: useless, drop it
    COUNT(DISTINCT status)         AS distinct_statuses, --3 statuses: in progress, closed, unspecified
    COUNT(DISTINCT incident_zip)     AS distinct_zip,
    COUNT(DISTINCT descriptor)     AS distinct_descriptors,
    COUNT(DISTINCT location_type)  AS distinct_location_types
FROM workspace.default.rat_sightings;

-- unusable columns: complaint_type, only one value which is rodent for all rows


distinct_complaint_types,distinct_statuses,distinct_zip,distinct_descriptors,distinct_location_types
1,3,190,4,26


In [0]:
%sql
-- check if there are any null values in these columns
SELECT
    SUM(CASE WHEN unique_key   IS NULL THEN 1 ELSE 0 END) AS null_unique_key,
    SUM(CASE WHEN created_date IS NULL THEN 1 ELSE 0 END) AS null_created,
    SUM(CASE WHEN closed_date  IS NULL THEN 1 ELSE 0 END) AS null_closed,      
    SUM(CASE WHEN incident_zip IS NULL THEN 1 ELSE 0 END) AS null_zip,
    SUM(CASE WHEN latitude     IS NULL THEN 1 ELSE 0 END) AS null_lat,         
    SUM(CASE WHEN longitude    IS NULL THEN 1 ELSE 0 END) AS null_long        
FROM workspace.default.rat_sightings;

null_unique_key,null_created,null_closed,null_zip,null_lat,null_long
0,0,2480,0,94,94


In [0]:
%sql
-- is incident_zip actually a valid 5-digit NYC ZIP?
SELECT incident_zip, COUNT(*) AS count
FROM workspace.default.rat_sightings
WHERE CAST(incident_zip AS STRING) NOT RLIKE '^(10[0-4]|11[0-4]|116)[0-9]{2}$'
GROUP BY incident_zip;
-- 12345 shows up once — not a real ZIP

incident_zip,count
12345,1


In [0]:
%sql
-- location_type: same category spelled two ways
SELECT location_type, COUNT(*) AS count
FROM workspace.default.rat_sightings
GROUP BY location_type
ORDER BY count DESC;
-- "Parking Lot or Garage" and "Parking Lot/Garage" are the same bucket, stored twice

location_type,count
3+ Family Apt. Building,25368
1-2 Family Dwelling,9879
3+ Family Mixed Use Building,3452
Other (Explain Below),3227
Sidewalk,2052
Commercial Building,1886
Street,1058
1-2 Family Mixed Use Building,799
Vacant Lot,768
Vacant Building,434


In [0]:
%sql
-- duplicate key check
SELECT COUNT(*) - COUNT(DISTINCT unique_key) AS duplicate_keys
FROM workspace.default.rat_sightings;


duplicate_keys
0


**Restaurant Inspections Data**

In [0]:
%sql
-- --- restaurant_inspections: row count -----------------------------------
SELECT COUNT(*) AS row_count FROM workspace.default.restaurant_inspections;
-- 158,083 — one row per VIOLATION, not per restaurant
-- some restaurants have more than one violation

row_count
158083


In [0]:
%sql
SELECT
COUNT(DISTINCT zipcode)     AS distinct_zip
FROM workspace.default.restaurant_inspections;


distinct_zip
232


In [0]:
%sql
-- --- restaurant_inspections: date range -----------------------------------
SELECT MIN(inspection_date) AS earliest, MAX(inspection_date) AS latest
FROM workspace.default.restaurant_inspections;
-- 2025-01-02 to 2026-09-16

earliest,latest
2025-01-02T00:00:00.000Z,2026-09-16T00:00:00.000Z


In [0]:
%sql
-- --- restaurant_inspections: CHECKPOINT — how many restaurants? ------------
SELECT COUNT(DISTINCT camis) AS distinct_restaurants
FROM workspace.default.restaurant_inspections;
-- 26,114 unique restaurants <-- checkpoint answer
-- NOT COUNT(*), which gives 158,083 rows, since some resturants have more than 1 violation

In [0]:
%sql
-- --- restaurant_inspections: which columns are unusable as delivered -----

SELECT
    SUM(CASE WHEN zipcode           IS NULL THEN 1 ELSE 0 END) AS null_zip,     -- 1,513
    SUM(CASE WHEN violation_code    IS NULL THEN 1 ELSE 0 END) AS null_vcode,   -- 1,803
    SUM(CASE WHEN score             IS NULL THEN 1 ELSE 0 END) AS null_score,   -- 7,889
    SUM(CASE WHEN grade             IS NULL THEN 1 ELSE 0 END) AS null_grade    -- 77,345 (48.9%)
FROM workspace.default.restaurant_inspections;

null_zip,null_vcode,null_score,null_grade
1513,1803,7889,77345


In [0]:
%sql
-- boro has a literal '0' where it should be a borough name
SELECT boro, COUNT(*) AS n
FROM workspace.default.restaurant_inspections
GROUP BY boro
ORDER BY n DESC;
-- 112 rows are boro = '0'

boro,n
Manhattan,58156
Queens,41879
Brooklyn,38613
Bronx,14785
Staten Island,4538
0,112


In [0]:
%sql
-- zipcode: out-of-state ZIPs hiding in an NYC file
SELECT
    LPAD(CAST(CAST(zipcode AS INT) AS STRING), 5, '0') AS zip5,
    COUNT(*) AS n
FROM workspace.default.restaurant_inspections
WHERE zipcode IS NOT NULL
  AND LPAD(CAST(CAST(zipcode AS INT) AS STRING), 5, '0') NOT RLIKE '^(10[0-4]|11[0-4]|116)[0-9]{2}$'
GROUP BY 1
ORDER BY n DESC;
-- 07307 (Jersey City), 08550 (Princeton Jct), 07631, 11701, 11542... 80 rows total, not NYC

zip5,n
07307,18
11701,15
08550,12
07631,12
07304,5
11542,4
07002,3
10550,3
07627,2
11762,2


In [0]:
%sql
-- grade column carries non-grade junk mixed with real grades
SELECT grade, COUNT(*) AS n
FROM workspace.default.restaurant_inspections
GROUP BY grade
ORDER BY n DESC;
-- A/B/C are real grades; N, Z, P are inspection-pending/other codes, not grades

grade,n
null,77345
A,50954
N,9248
B,8875
C,6504
Z,4874
P,283


In [0]:
%sql
-- exact duplicate rows
SELECT COUNT(*) AS n FROM (
    SELECT camis, dba, boro, zipcode, cuisine_description, inspection_date,
           violation_code, violation_description, critical_flag, score, grade,
           COUNT(*) AS c
    FROM workspace.default.restaurant_inspections
    GROUP BY 1,2,3,4,5,6,7,8,9,10,11
    HAVING COUNT(*) > 1
);
-- 166 exact duplicate rows

n
166


In [0]:
%sql
-- confirm one ZIP per restaurant (safe join key)
SELECT COUNT(*) AS restaurants_with_multiple_zips FROM (
    SELECT camis FROM workspace.default.restaurant_inspections
    WHERE zipcode IS NOT NULL
    GROUP BY camis
    HAVING COUNT(DISTINCT zipcode) > 1
);
-- 0 — camis maps to exactly one zipcode

restaurants_with_multiple_zips
0


In [0]:
%sql
-- =====================================================================
-- BLOCK TWO — CLEAN AND JOIN
-- =====================================================================

-- ---------------------------------------------------------------------
-- 2.1  Cleaned table: rat_sightings, ZIP-normalized, rodent complaint
--      decision applied and written into the columns themselves.
--
--      DECISION (write this down, say it out loud at demo):
--      A rodent complaint = every row (complaint_type is always
--      "Rodent"). We split into two classes by descriptor:
--        EVIDENCE   = Rat Sighting, Mouse Sighting, Signs of Rodents
--                     -> someone reported actually seeing a rodent
--                        or its droppings
--        CONDITION  = Condition Attracting Rodents
--                     -> someone reported garbage/harborage; no
--                        rodent was seen
--      Mice are kept in EVIDENCE because the 311 complaint_type is
--      "Rodent," not "Rat" — dropping mice would silently discard
--      2,046 real service requests. CONDITION rows are never counted
--      as evidence a rodent was present.
-- ---------------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.default.rodent_complaints_clean AS
SELECT
    CAST(unique_key AS BIGINT)                              AS complaint_id,
    LPAD(CAST(CAST(incident_zip AS INT) AS STRING), 5, '0') AS zip_code,
    UPPER(TRIM(borough))                                    AS borough,
    CAST(created_date AS TIMESTAMP)                         AS created_ts,
    CAST(closed_date  AS TIMESTAMP)                         AS closed_ts,
    TRIM(status)                                            AS status,
    TRIM(descriptor)                                        AS descriptor,
    -- collapse the duplicate-spelling category noticed in Block One
    CASE WHEN TRIM(location_type) IN ('Parking Lot or Garage','Parking Lot/Garage')
         THEN 'Parking Lot/Garage' ELSE TRIM(location_type) END AS location_type,
    CASE WHEN TRIM(descriptor) IN ('Rat Sighting','Mouse Sighting','Signs of Rodents')
         THEN TRUE ELSE FALSE END                           AS is_rodent_evidence,
    CASE WHEN TRIM(descriptor) = 'Condition Attracting Rodents'
         THEN TRUE ELSE FALSE END                           AS is_condition_report,
    CASE WHEN closed_date IS NOT NULL THEN TRUE ELSE FALSE END AS is_closed,
    CASE WHEN closed_date IS NOT NULL
         THEN (UNIX_TIMESTAMP(CAST(closed_date AS TIMESTAMP))
             - UNIX_TIMESTAMP(CAST(created_date AS TIMESTAMP))) / 3600.0
    END                                                      AS hours_to_close
FROM workspace.default.rat_sightings
WHERE incident_zip IS NOT NULL
  AND LPAD(CAST(CAST(incident_zip AS INT) AS STRING), 5, '0')
      RLIKE '^(10[0-4]|11[0-4]|116)[0-9]{2}$'
  AND created_date IS NOT NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.rodent_complaints_clean;  -- expect 50,953

COUNT(*)
50953


In [0]:
%sql
-- ---------------------------------------------------------------------
-- 2.2  Cleaned table: restaurant_inspections, ZIP-normalized.
--      04K = rat evidence, 04L = mouse evidence, 08A = harborage
--      condition (the inspection-side analogue of the 311 condition
--      complaint — kept separate from evidence, same logic as above).
-- ---------------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.default.inspection_violations_clean AS
SELECT DISTINCT
    CAST(camis AS BIGINT)                                    AS camis,
    TRIM(dba)                                                AS restaurant_name,
    LPAD(CAST(CAST(zipcode AS INT) AS STRING), 5, '0')       AS zip_code,
    CASE WHEN TRIM(boro) IN ('Manhattan','Brooklyn','Queens','Bronx','Staten Island')
         THEN UPPER(TRIM(boro)) END                          AS borough,
    TRIM(cuisine_description)                                AS cuisine,
    CAST(inspection_date AS TIMESTAMP)                       AS inspection_ts,
    TRIM(violation_code)                                     AS violation_code,
    TRIM(violation_description)                              AS violation_description,
    TRIM(critical_flag)                                      AS critical_flag,
    CAST(score AS DOUBLE)                                    AS score,
    CASE WHEN TRIM(grade) IN ('A','B','C') THEN TRIM(grade) END AS grade,
    CASE WHEN TRIM(violation_code) IN ('04K','04L') THEN TRUE ELSE FALSE END AS is_rodent_evidence,
    CASE WHEN TRIM(violation_code) = '04K' THEN TRUE ELSE FALSE END          AS is_rat_evidence,
    CASE WHEN TRIM(violation_code) = '04L' THEN TRUE ELSE FALSE END         AS is_mouse_evidence,
    CASE WHEN TRIM(violation_code) = '08A' THEN TRUE ELSE FALSE END         AS is_harborage
FROM workspace.default.restaurant_inspections
WHERE zipcode IS NOT NULL
  AND LPAD(CAST(CAST(zipcode AS INT) AS STRING), 5, '0')
      RLIKE '^(10[0-4]|11[0-4]|116)[0-9]{2}$'
  AND camis IS NOT NULL;
  -- DISTINCT drops the 166 exact duplicate rows found in Block One

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS rows, COUNT(DISTINCT camis) AS restaurants
FROM workspace.default.inspection_violations_clean;
-- rows drop from 158,083 to ~156,300ish after excluding non-NYC ZIPs + dupes;
-- restaurants stay close to 26,114 minus whichever were NJ/NY-suburb only

rows,restaurants
156324,25779


In [0]:
%sql
-- ---------------------------------------------------------------------
-- 2.3  Index table: zip_service_gap — one row per ZIP, the JOIN
-- ---------------------------------------------------------------------

CREATE OR REPLACE TABLE workspace.default.zip_service_gap AS
WITH complaints AS (
    SELECT
        zip_code,
        COUNT(*)                                             AS complaints_total,
        SUM(CASE WHEN is_rodent_evidence  THEN 1 ELSE 0 END) AS complaints_evidence,
        SUM(CASE WHEN is_condition_report THEN 1 ELSE 0 END) AS complaints_condition,
        AVG(CASE WHEN is_closed THEN 1.0 ELSE 0.0 END)       AS pct_closed,
        PERCENTILE_APPROX(hours_to_close, 0.5)               AS median_hours_to_close,
        AVG(CASE WHEN hours_to_close IS NOT NULL AND hours_to_close < 1 THEN 1.0 ELSE 0.0 END)
                                                              AS pct_closed_under_1hr
    FROM workspace.default.rodent_complaints_clean
    GROUP BY zip_code
),
boro AS (
    SELECT zip_code, borough FROM (
        SELECT zip_code, borough,
               ROW_NUMBER() OVER (PARTITION BY zip_code ORDER BY COUNT(*) DESC) AS rn
        FROM workspace.default.rodent_complaints_clean
        WHERE borough IS NOT NULL AND borough <> 'UNSPECIFIED'
        GROUP BY zip_code, borough
    ) t WHERE rn = 1
),
inspections AS (
    SELECT
        zip_code,
        COUNT(DISTINCT camis)                                             AS restaurants_inspected,
        COUNT(DISTINCT CASE WHEN is_rodent_evidence THEN camis END)       AS restaurants_rodent_evidence,
        COUNT(DISTINCT CASE WHEN is_harborage       THEN camis END)       AS restaurants_harborage
    FROM workspace.default.inspection_violations_clean
    GROUP BY zip_code
),
joined AS (
    SELECT
        COALESCE(c.zip_code, i.zip_code)               AS zip_code,
        b.borough,
        COALESCE(c.complaints_total, 0)                AS complaints_total,
        COALESCE(c.complaints_evidence, 0)             AS complaints_evidence,
        COALESCE(c.complaints_condition, 0)            AS complaints_condition,
        c.pct_closed, c.median_hours_to_close, c.pct_closed_under_1hr,
        COALESCE(i.restaurants_inspected, 0)           AS restaurants_inspected,
        COALESCE(i.restaurants_rodent_evidence, 0)     AS restaurants_rodent_evidence,
        COALESCE(i.restaurants_harborage, 0)           AS restaurants_harborage
    FROM complaints c
    FULL OUTER JOIN inspections i ON c.zip_code = i.zip_code
    LEFT JOIN boro b ON b.zip_code = COALESCE(c.zip_code, i.zip_code)
),
rated AS (
    SELECT *,
        CASE WHEN restaurants_inspected >= 1
             THEN restaurants_rodent_evidence * 1.0 / restaurants_inspected END AS verified_rodent_rate,
        CASE WHEN restaurants_inspected >= 1
             THEN complaints_total * 100.0 / restaurants_inspected END          AS complaints_per_100_restaurants,
        CASE
            WHEN restaurants_inspected >= 50 AND complaints_total >= 50 THEN 'sufficient'
            WHEN restaurants_inspected >= 15 AND complaints_total >= 15 THEN 'thin'
            ELSE 'insufficient'
        END                                                                     AS data_sufficiency
    FROM joined
),
stats AS (
    SELECT
        AVG(verified_rodent_rate)           AS mu_ver, STDDEV(verified_rodent_rate)           AS sd_ver,
        AVG(complaints_per_100_restaurants) AS mu_rep, STDDEV(complaints_per_100_restaurants) AS sd_rep
    FROM rated WHERE data_sufficiency = 'sufficient'
),
scored AS (
    SELECT r.*,
        (r.verified_rodent_rate           - s.mu_ver) / NULLIF(s.sd_ver, 0) AS z_verified,
        (r.complaints_per_100_restaurants - s.mu_rep) / NULLIF(s.sd_rep, 0) AS z_reporting
    FROM rated r CROSS JOIN stats s
)
SELECT
    zip_code, borough,
    complaints_total, complaints_evidence, complaints_condition,
    restaurants_inspected, restaurants_rodent_evidence, restaurants_harborage,
    ROUND(verified_rodent_rate, 4)           AS verified_rodent_rate,
    ROUND(complaints_per_100_restaurants, 2) AS complaints_per_100_restaurants,
    ROUND(pct_closed, 4)                     AS pct_closed,
    ROUND(median_hours_to_close, 1)          AS median_hours_to_close,
    ROUND(pct_closed_under_1hr, 4)           AS pct_closed_under_1hr,
    ROUND(z_verified, 3)                     AS z_verified,
    ROUND(z_reporting, 3)                    AS z_reporting,
    CASE WHEN data_sufficiency = 'insufficient' THEN NULL
         ELSE ROUND(z_verified - z_reporting, 3) END AS service_gap_index,
    data_sufficiency,
    CASE
      WHEN data_sufficiency = 'insufficient'
        THEN CONCAT('Not enough data: ', CAST(restaurants_inspected AS STRING),
                    ' restaurants, ', CAST(complaints_total AS STRING), ' complaints.')
      WHEN data_sufficiency = 'thin' THEN 'Low volume — index unstable, treat as indicative only.'
      ELSE 'Sufficient data.'
    END                                       AS zip_note
FROM scored
ORDER BY zip_code;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- checks: run these, don't trust the build
SHOW TABLES IN workspace.default;

In [0]:
%sql
SELECT COUNT(*) AS zips FROM workspace.default.zip_service_gap;

In [0]:
%sql
SELECT data_sufficiency, COUNT(*) FROM workspace.default.zip_service_gap GROUP BY 1;

In [0]:
%sql
SELECT * FROM workspace.default.zip_service_gap WHERE zip_code = '11430';  -- JFK, should be insufficient

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 2.4  Descriptions — every table, every column. Genie reads these.
--      Re-run this whole section after ANY CREATE OR REPLACE above —
--      rebuilding a table wipes its comments.
-- ---------------------------------------------------------------------

COMMENT ON TABLE workspace.default.rodent_complaints_clean IS
'One row per 311 rodent complaint, Jan 2025 – Sep 2026, NYC ZIPs only. ZIP code and location_type refer to the reported incident location (where the rodent sighting or condition was observed), not the caller''s home address. Descriptor splits into rodent sightings (is_rodent_evidence) and reports of conditions attracting rodents (is_condition_report). These are citizen-reported complaints, not a systematic survey of rodent populations — complaint volume reflects reporting behavior as much as actual rodent activity.';

In [0]:
%sql
COMMENT ON TABLE workspace.default.inspection_violations_clean IS
'One row per restaurant health violation, Jan 2025 - Sep 2026, NYC ZIPs only, deduplicated. One restaurant cited for eight things is eight rows — always COUNT(DISTINCT camis) for restaurant counts. Rodent codes: 04K rats, 04L mice, 08A harborage conditions.';

In [0]:
%sql
COMMENT ON TABLE workspace.default.zip_service_gap IS
'One row per NYC ZIP code. Combines 311 rodent complaints (resident-reported) with DOHMH inspection violations (inspector-verified) into a service_gap_index. This is NOT a rat population estimate and must not be described as ranking the worst rat neighborhoods.';

In [0]:
%sql
-- rodent_complaints_clean column comments
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN complaint_id COMMENT 'Original 311 unique_key, cast to BIGINT. Primary key for this table.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN zip_code COMMENT 'Five-digit NYC ZIP, zero-padded string. Non-NYC and null ZIPs already excluded from this table.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN borough COMMENT 'Borough as reported on the 311 complaint.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN created_ts COMMENT 'Timestamp the 311 complaint was filed.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN closed_ts COMMENT 'Timestamp the complaint was closed. Null means still open.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN status COMMENT 'Current 311 status: Closed, In Progress, or Unspecified.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN descriptor COMMENT 'Original 311 descriptor: Rat Sighting, Mouse Sighting, Signs of Rodents, or Condition Attracting Rodents.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN location_type COMMENT 'Where the complaint occurred. "Parking Lot or Garage" and "Parking Lot/Garage" collapsed into one label.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN is_rodent_evidence COMMENT 'TRUE if the resident reported actually seeing a rodent or its signs (Rat Sighting, Mouse Sighting, Signs of Rodents).';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN is_condition_report COMMENT 'TRUE if the complaint was about conditions attracting rodents (garbage/harborage), with no rodent reported seen.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN is_closed COMMENT 'TRUE if closed_ts is not null.';
ALTER TABLE workspace.default.rodent_complaints_clean ALTER COLUMN hours_to_close COMMENT 'Hours from created_ts to closed_ts. Null for open complaints. Median citywide is under 7 hours — most complaints close on intake, not after a site visit.';

In [0]:
%sql
-- inspection_violations_clean column comments
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN camis COMMENT 'NYC restaurant permit ID. Use COUNT(DISTINCT camis) for restaurant counts — this table is one row per violation.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN restaurant_name COMMENT 'Restaurant name (dba) at time of inspection.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN zip_code COMMENT 'Five-digit NYC ZIP, zero-padded string. Non-NYC ZIPs (NJ, Nassau, Westchester) already excluded.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN borough COMMENT 'Borough name, nulled where the source had the literal value "0".';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN cuisine COMMENT 'Self-reported cuisine category.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN inspection_ts COMMENT 'Date of this inspection.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN violation_code COMMENT 'DOHMH violation code. Null means the inspection recorded no violation (critical_flag = Not Applicable).';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN violation_description COMMENT 'Text description of the violation code.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN critical_flag COMMENT 'Critical, Not Critical, or Not Applicable (no violation cited).';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN score COMMENT 'Inspection point score; higher is worse. Null for ~5% of rows.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN grade COMMENT 'Letter grade A/B/C only. Non-grade codes (N, Z, P) from the source and the 49% null rate have been excluded — treat absence as "no grade posted," not as a failing grade.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN is_rodent_evidence COMMENT 'TRUE for violation code 04K (rats) or 04L (mice) — an inspector directly observed evidence.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN is_rat_evidence COMMENT 'TRUE for 04K specifically: evidence of rats or live rats.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN is_mouse_evidence COMMENT 'TRUE for 04L specifically: evidence of mice or live mice.';
ALTER TABLE workspace.default.inspection_violations_clean ALTER COLUMN is_harborage COMMENT 'TRUE for 08A: conditions conducive to rodents/insects. A condition, not proof a rodent was present — the inspection-side analogue of a 311 condition complaint.';

In [0]:
%sql
-- zip_service_gap column comments
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN zip_code COMMENT 'Five-digit NYC ZIP. Primary key, one row per ZIP.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN borough COMMENT 'Modal borough for this ZIP, from its 311 complaints.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN complaints_total COMMENT 'All 311 rodent complaints in this ZIP (evidence + condition). Measures reporting behavior, not rodent presence.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN complaints_evidence COMMENT 'Complaints where a rodent or its signs were reported seen.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN complaints_condition COMMENT 'Complaints about conditions attracting rodents; no rodent reported seen.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN restaurants_inspected COMMENT 'Distinct restaurants inspected in this ZIP. Denominator for every rate below.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN restaurants_rodent_evidence COMMENT 'Distinct restaurants cited 04K or 04L — inspector-verified rodent evidence, independent of 311 calls.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN restaurants_harborage COMMENT 'Distinct restaurants cited 08A, conditions conducive to rodents.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN verified_rodent_rate COMMENT 'restaurants_rodent_evidence / restaurants_inspected. Best available proxy for real rodent conditions since it does not depend on residents calling.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN complaints_per_100_restaurants COMMENT '311 complaints per 100 inspected restaurants — density-adjusted reporting intensity.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN pct_closed COMMENT 'Share of this ZIP''s complaints with a closed_ts.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN median_hours_to_close COMMENT 'Median hours from filing to closure among closed complaints. Citywide median is under 7 hours — most closures happen on intake, not after a site visit.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN pct_closed_under_1hr COMMENT 'Share of complaints closed within one hour of filing — flags likely auto-closures rather than serviced requests.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN z_verified COMMENT 'verified_rodent_rate as a z-score across ZIPs with sufficient data.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN z_reporting COMMENT 'complaints_per_100_restaurants as a z-score across ZIPs with sufficient data.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN service_gap_index COMMENT 'z_verified minus z_reporting. High = inspectors verify rodent conditions here but residents report relatively little. NOT a rat population estimate; null where data_sufficiency = insufficient.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN data_sufficiency COMMENT 'sufficient (50+ restaurants and 50+ complaints), thin (15+ of each), or insufficient. Index withheld for insufficient ZIPs so low-volume areas like airports cannot rank as best or worst.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN zip_note COMMENT 'Plain-language caveat for this ZIP, meant for display when a user filters the dashboard to it.';

# BLOCK THREE — THE HIDDEN ISSUE

The `service_gap_index` in the gold table produces conclusions that are the **opposite of the truth**.

The index compares `z_verified` (inspector-verified restaurant rodent evidence)
against `z_reporting` (311 complaints per 100 restaurants). It assumes that a ZIP
with high verified rodent rates but low complaint rates has a "service gap" —
residents aren't reporting a real problem.

But the 311 complaints are **not about restaurants**. They are overwhelmingly
about **residential buildings** — apartment complexes, 1-2 family dwellings,
sidewalks, and vacant lots. Dividing residential rodent complaints by the number
of restaurants in a ZIP creates a **numerator-denominator mismatch** that
inverts the ranking:

* Dense residential ZIPs with many rodent complaints but few restaurants get
  extreme `complaints_per_100_restaurants` values → high `z_reporting` →
  **negative** service gap → labeled "over-reporters."
* Commercial ZIPs with more restaurants and fewer residential complaints get
  low `complaints_per_100_restaurants` → low `z_reporting` →
  **positive** service gap → labeled "under-served."

The ZIPs flagged as "over-reporters" actually have **the most rodent complaints
in the city** — they are the neighborhoods with the worst rodent problems.
The index would direct city resources **away** from them.


In [0]:
%sql
-- ---------------------------------------------------------------------
-- 3.1  QUERY 1 — What ARE these 311 complaints actually about?
--      Stop and ask: what is a "rat sighting" record? It is a citizen
--      phone call to 311 reporting a rodent at a specific location.
--      That location is almost never a restaurant.
-- ---------------------------------------------------------------------
SELECT location_type, COUNT(*) AS n,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM workspace.default.rodent_complaints_clean
GROUP BY location_type
ORDER BY n DESC;

-- RESULT: ~70% of complaints are about 3+ Family Apt. Buildings,
-- 1-2 Family Dwellings, and Mixed-Use buildings. "Commercial Building"
-- (which would include restaurants) is 3.7%. These are residential
-- rodent complaints, not restaurant complaints.

In [0]:
%sql
-- ---------------------------------------------------------------------
-- 3.2  QUERY 2 — Do the two metrics even measure the same thing?
--      If 311 complaints are about residential buildings and verified
--      rodent evidence is about restaurants, the correlation should
--      be near zero — they are measuring different phenomena.
-- ---------------------------------------------------------------------
SELECT
  CORR(complaints_per_100_restaurants, verified_rodent_rate) AS r_per100_vs_verified,
  CORR(complaints_total,             verified_rodent_rate) AS r_total_vs_verified,
  CORR(complaints_total,             restaurants_inspected) AS r_complaints_vs_restaurants,
  COUNT(*) AS n
FROM workspace.default.zip_service_gap
WHERE data_sufficiency = 'sufficient';

-- RESULT: r_per100_vs_verified = 0.19 (weak), r_total_vs_verified = 0.14 (weaker),
-- r_complaints_vs_restaurants = -0.03 (none). The number of restaurants in a ZIP
-- is unrelated to how many rodent complaints it generates. Dividing one by the
-- other is meaningless.

r_per100_vs_verified,r_total_vs_verified,r_complaints_vs_restaurants,n
0.1877223802652922,0.14176421230239522,-0.032390344223215135,125


In [0]:
%sql
-- ---------------------------------------------------------------------
-- 3.3  QUERY 3 — Does the service_gap_index invert the truth?
--      Compare ZIPs flagged as "service gap" (index > 1, supposedly
--      under-reporting a real problem) against ZIPs flagged as
--      "over-reporting" (index < -1, supposedly crying wolf).
--      Look at TOTAL rodent complaints — the actual volume of suffering.
-- ---------------------------------------------------------------------
SELECT
  CASE WHEN service_gap_index >  1 THEN 'Flagged: Service Gap (under-reporting)'
       WHEN service_gap_index < -1 THEN 'Flagged: Over-Reporting'
       ELSE 'Middle' END AS index_label,
  COUNT(*)                            AS n_zips,
  SUM(complaints_total)               AS total_rodent_complaints,
  ROUND(AVG(verified_rodent_rate), 4) AS avg_restaurant_rodent_rate,
  SUM(restaurants_inspected)          AS total_restaurants
FROM workspace.default.zip_service_gap
WHERE data_sufficiency = 'sufficient'
GROUP BY 1
ORDER BY total_rodent_complaints DESC;

-- RESULT: "Over-Reporting" ZIPs have 12,050 total rodent complaints.
--         "Service Gap" ZIPs have only 5,868.
--         The index labels the neighborhoods with 2x MORE rodent complaints
--         as over-reporters, and the neighborhoods with fewer complaints as
--         having a hidden service gap. The conclusion is backwards.

index_label,n_zips,total_rodent_complaints,avg_restaurant_rodent_rate,total_restaurants
Middle,80,28524,0.2349,17234
Flagged: Over-Reporting,21,12050,0.2006,2703
Flagged: Service Gap (under-reporting),24,5868,0.3451,3256


In [0]:
%sql
-- ---------------------------------------------------------------------
-- 3.4  QUERY 4 — Decile view: as the service_gap_index rises,
--      total complaints FALL. The index ranks the quietest ZIPs as
--      the worst service gaps.
-- ---------------------------------------------------------------------
WITH ranked AS (
  SELECT *,
    NTILE(10) OVER (ORDER BY service_gap_index ASC) AS gap_decile
  FROM workspace.default.zip_service_gap
  WHERE data_sufficiency = 'sufficient'
)
SELECT
  gap_decile,
  COUNT(*)                              AS n_zips,
  ROUND(AVG(verified_rodent_rate), 4)    AS avg_verified_rate,
  SUM(complaints_total)                  AS total_complaints,
  SUM(restaurants_inspected)            AS total_restaurants,
  ROUND(AVG(complaints_per_100_restaurants), 1) AS avg_complaints_per_100
FROM ranked
GROUP BY gap_decile
ORDER BY gap_decile;

-- RESULT: Decile 1 (most negative index — "over-reporters"): 9,236 complaints.
--         Decile 10 (most positive index — "service gap"):   2,477 complaints.
--         The ZIPs with the most rodent complaints are at the bottom of the
--         index. The ZIPs with the fewest are at the top.

gap_decile,n_zips,avg_verified_rate,total_complaints,total_restaurants,avg_complaints_per_100
1,13,0.2193,9236,1259,765.1
2,13,0.1776,4804,2620,285.5
3,13,0.2088,5958,1998,290.3
4,13,0.2073,4637,2972,209.6
5,13,0.2289,3647,2463,201.8
6,12,0.2430,4258,3457,186.0
7,12,0.2723,4852,2743,229.4
8,12,0.2730,3182,2425,155.4
9,12,0.3183,3391,1759,216.0
10,12,0.3719,2477,1497,173.6


## What the index gets backwards

| Metric | "Over-Reporting" ZIPs (index < -1) | "Service Gap" ZIPs (index > 1) |
|---|---|---|
| ZIP count | 21 | 24 |
| **Total rodent complaints** | **12,050** | **5,868** |
| Avg restaurant rodent rate | 20.1% | 34.5% |
| Total restaurants | 2,703 | 3,256 |

The "service gap" ZIPs do have a higher **restaurant** rodent rate (34.5% vs
20.1%) — inspectors find more rodent evidence in their restaurants. But those
ZIPs have **fewer than half** the total rodent complaints. The rodent problem in
the "over-reporting" ZIPs is real, severe, and located in residential buildings
— exactly where restaurant inspectors never look.

### The careless conclusion (wrong)
> "ZIPs with a high service gap index have hidden rodent problems that residents
> aren't reporting. Direct outreach and inspection resources to these ZIPs."

### The truth (opposite)
> The ZIPs flagged as 'over-reporting' have the most rodent complaints in the
city — 12,050 vs 5,868. They are dense residential neighborhoods (East Harlem,
South Bronx, Central Brooklyn) where rats infest apartment buildings and
sidewalks, not restaurants. The index penalizes them for having few restaurants
relative to their residential complaint volume, and would divert city
resources **away** from the neighborhoods that need them most.

### Root cause
`complaints_per_100_restaurants` divides a **residential** numerator by a
**commercial** denominator. The number of restaurants in a ZIP is uncorrelated
with its rodent complaint volume (r = -0.03). The ratio is noise, not signal.

### What would fix it
A valid reporting-intensity metric would divide 311 complaints by a denominator
that matches the complaint population — residential units, population, or
total addresses — not restaurant count. Until then, `service_gap_index` must
not be used to rank or prioritize neighborhoods.

In [0]:
%sql
-- =====================================================================
-- BLOCK THREE FIX — rebuild zip_service_gap without the numerator/
-- denominator mismatch found in the hidden-issue queries (3.1-3.4).
--
-- ROOT CAUSE: complaints_per_100_restaurants divided a RESIDENTIAL
-- numerator (70% of 311 rodent complaints are apartment/dwelling) by a
-- COMMERCIAL denominator (restaurant count), which are uncorrelated
-- (r = -0.03). Forcing one composite score across two unrelated
-- populations inverted the ranking.
--
-- FIX: split into two honest, narrow metrics instead of one wrong one.
--   1. restaurant_service_gap_index — commercial complaints vs verified
--      restaurant rodent evidence. Same population (restaurants) on
--      both sides. This is the ONLY valid place for a restaurant
--      denominator.
--   2. residential_rodent_rank — raw volume/rank of residential rodent
--      complaints. No restaurant denominator. Flagged as needing a
--      population denominator we don't have in either source file.
-- The old service_gap_index is kept, renamed, and marked invalid so
-- the audit trail of what was tried and rejected is visible.
-- =====================================================================

CREATE OR REPLACE TABLE workspace.default.zip_service_gap AS
WITH complaints AS (
    SELECT
        zip_code,
        COUNT(*)                                                        AS complaints_total,
        SUM(CASE WHEN is_rodent_evidence  THEN 1 ELSE 0 END)            AS complaints_evidence,
        SUM(CASE WHEN is_condition_report THEN 1 ELSE 0 END)            AS complaints_condition,
        -- NEW: split by what kind of building the complaint is about
        SUM(CASE WHEN location_type = 'Commercial Building' THEN 1 ELSE 0 END) AS complaints_commercial,
        SUM(CASE WHEN location_type <> 'Commercial Building' OR location_type IS NULL
                 THEN 1 ELSE 0 END)                                      AS complaints_residential,
        AVG(CASE WHEN is_closed THEN 1.0 ELSE 0.0 END)                   AS pct_closed,
        PERCENTILE_APPROX(hours_to_close, 0.5)                          AS median_hours_to_close,
        AVG(CASE WHEN hours_to_close IS NOT NULL AND hours_to_close < 1 THEN 1.0 ELSE 0.0 END)
                                                                          AS pct_closed_under_1hr
    FROM workspace.default.rodent_complaints_clean
    GROUP BY zip_code
),
boro AS (
    SELECT zip_code, borough FROM (
        SELECT zip_code, borough,
               ROW_NUMBER() OVER (PARTITION BY zip_code ORDER BY COUNT(*) DESC) AS rn
        FROM workspace.default.rodent_complaints_clean
        WHERE borough IS NOT NULL AND borough <> 'UNSPECIFIED'
        GROUP BY zip_code, borough
    ) t WHERE rn = 1
),
inspections AS (
    SELECT
        zip_code,
        COUNT(DISTINCT camis)                                       AS restaurants_inspected,
        COUNT(DISTINCT CASE WHEN is_rodent_evidence THEN camis END) AS restaurants_rodent_evidence,
        COUNT(DISTINCT CASE WHEN is_harborage       THEN camis END) AS restaurants_harborage
    FROM workspace.default.inspection_violations_clean
    GROUP BY zip_code
),
joined AS (
    SELECT
        COALESCE(c.zip_code, i.zip_code)             AS zip_code,
        b.borough,
        COALESCE(c.complaints_total, 0)              AS complaints_total,
        COALESCE(c.complaints_evidence, 0)           AS complaints_evidence,
        COALESCE(c.complaints_condition, 0)          AS complaints_condition,
        COALESCE(c.complaints_commercial, 0)         AS complaints_commercial,
        COALESCE(c.complaints_residential, 0)        AS complaints_residential,
        c.pct_closed, c.median_hours_to_close, c.pct_closed_under_1hr,
        COALESCE(i.restaurants_inspected, 0)         AS restaurants_inspected,
        COALESCE(i.restaurants_rodent_evidence, 0)   AS restaurants_rodent_evidence,
        COALESCE(i.restaurants_harborage, 0)         AS restaurants_harborage
    FROM complaints c
    FULL OUTER JOIN inspections i ON c.zip_code = i.zip_code
    LEFT JOIN boro b ON b.zip_code = COALESCE(c.zip_code, i.zip_code)
),
rated AS (
    SELECT *,
        CASE WHEN restaurants_inspected >= 1
             THEN restaurants_rodent_evidence * 1.0 / restaurants_inspected END AS verified_rodent_rate,
        -- OLD, INVALID metric — kept only as an audit trail, never displayed as a real score
        CASE WHEN restaurants_inspected >= 1
             THEN complaints_total * 100.0 / restaurants_inspected END          AS invalid_complaints_per_100_restaurants,
        -- NEW, VALID metric — commercial complaints over restaurant count.
        -- Same population (restaurants) on both sides of the ratio.
        CASE WHEN restaurants_inspected >= 1
             THEN complaints_commercial * 100.0 / restaurants_inspected END     AS commercial_complaints_per_100_restaurants,
        CASE
            WHEN restaurants_inspected >= 50 AND complaints_total >= 50 THEN 'sufficient'
            WHEN restaurants_inspected >= 15 AND complaints_total >= 15 THEN 'thin'
            ELSE 'insufficient'
        END                                                                     AS data_sufficiency
    FROM joined
),
stats AS (
    -- z-scores computed only from the corrected, apples-to-apples pair
    SELECT
        AVG(verified_rodent_rate)                    AS mu_ver, STDDEV(verified_rodent_rate)                    AS sd_ver,
        AVG(commercial_complaints_per_100_restaurants) AS mu_rep, STDDEV(commercial_complaints_per_100_restaurants) AS sd_rep
    FROM rated WHERE data_sufficiency = 'sufficient'
),
scored AS (
    SELECT r.*,
        (r.verified_rodent_rate                      - s.mu_ver) / NULLIF(s.sd_ver, 0) AS z_verified,
        (r.commercial_complaints_per_100_restaurants  - s.mu_rep) / NULLIF(s.sd_rep, 0) AS z_reporting_commercial,
        -- residential complaint rank, city-wide, among sufficient ZIPs.
        -- No restaurant denominator. 1 = fewest residential complaints,
        -- N = most. Rank, not a rate, because we have no population
        -- denominator in either source file.
        RANK() OVER (ORDER BY r.complaints_residential DESC) AS residential_complaint_rank_citywide
    FROM rated r CROSS JOIN stats s
)
SELECT
    zip_code, borough,
    complaints_total, complaints_evidence, complaints_condition,
    complaints_commercial, complaints_residential,
    restaurants_inspected, restaurants_rodent_evidence, restaurants_harborage,
    ROUND(verified_rodent_rate, 4)                         AS verified_rodent_rate,
    ROUND(commercial_complaints_per_100_restaurants, 2)    AS commercial_complaints_per_100_restaurants,
    ROUND(pct_closed, 4)                                   AS pct_closed,
    ROUND(median_hours_to_close, 1)                        AS median_hours_to_close,
    ROUND(pct_closed_under_1hr, 4)                         AS pct_closed_under_1hr,
    ROUND(z_verified, 3)                                   AS z_verified,
    ROUND(z_reporting_commercial, 3)                       AS z_reporting_commercial,
    CASE WHEN data_sufficiency = 'insufficient' THEN NULL
         ELSE ROUND(z_verified - z_reporting_commercial, 3) END AS restaurant_service_gap_index,
    residential_complaint_rank_citywide,
    ROUND(invalid_complaints_per_100_restaurants, 2)       AS deprecated_invalid_ratio_DO_NOT_USE,
    data_sufficiency,
    CASE
      WHEN data_sufficiency = 'insufficient'
        THEN CONCAT('Not enough data: ', CAST(restaurants_inspected AS STRING),
                    ' restaurants, ', CAST(complaints_total AS STRING), ' complaints.')
      WHEN data_sufficiency = 'thin' THEN 'Low volume — index unstable, treat as indicative only.'
      ELSE 'Sufficient data.'
    END                                                      AS zip_note
FROM scored
ORDER BY zip_code;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- sanity check: confirm the inversion is gone for the restaurant-only metric
-- and that residential rank now tracks residential complaint volume directly
SELECT
  CASE WHEN restaurant_service_gap_index >  1 THEN 'High restaurant gap'
       WHEN restaurant_service_gap_index < -1 THEN 'Low restaurant gap'
       ELSE 'Middle' END AS label,
  COUNT(*) AS n_zips,
  SUM(complaints_commercial) AS total_commercial_complaints,   -- should track WITH the label now
  ROUND(AVG(verified_rodent_rate), 4) AS avg_restaurant_rate
FROM workspace.default.zip_service_gap
WHERE data_sufficiency = 'sufficient'
GROUP BY 1;

label,n_zips,total_commercial_complaints,avg_restaurant_rate
Low restaurant gap,18,360,0.1822
Middle,84,1144,0.2401
High restaurant gap,23,196,0.3408


In [0]:
%sql
-- =====================================================================
-- Updated / added column comments — re-run after this rebuild
-- =====================================================================

COMMENT ON TABLE workspace.default.zip_service_gap IS
'One row per NYC ZIP code. Splits rodent measurement into two valid, separately-scoped metrics after discovering that dividing residential 311 complaints by restaurant count inverted rankings (see Block Three notebook section): restaurant_service_gap_index compares commercial-only complaints to verified restaurant rodent evidence (same population, restaurants, on both sides); residential_complaint_rank_citywide ranks residential complaint volume directly with no restaurant denominator. The deprecated_invalid_ratio_DO_NOT_USE column is kept only as an audit trail of the original, incorrect metric.';

ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN complaints_commercial COMMENT
'311 rodent complaints where location_type = Commercial Building. The only complaint subset that shares a population with the restaurant inspection data.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN complaints_residential COMMENT
'311 rodent complaints at any non-commercial location (apartment buildings, 1-2 family dwellings, sidewalks, vacant lots, etc.) — roughly 96% of all rodent complaints. Has no restaurant denominator; do not divide by restaurants_inspected.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN commercial_complaints_per_100_restaurants COMMENT
'complaints_commercial / restaurants_inspected * 100. Valid ratio: both numerator and denominator describe the restaurant/commercial population in this ZIP.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN restaurant_service_gap_index COMMENT
'z_verified minus z_reporting_commercial. Scoped ONLY to restaurants: high value means inspectors verify rodent evidence in this ZIP''s restaurants more than commercial 311 complaints would suggest. Says nothing about residential rodent conditions.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN residential_complaint_rank_citywide COMMENT
'Citywide rank of this ZIP by residential rodent complaint volume (1 = most complaints). Raw rank, not a rate, because no population or housing-unit denominator is available in the source files. A per-capita version would need Census ZCTA population joined in separately.';
ALTER TABLE workspace.default.zip_service_gap ALTER COLUMN deprecated_invalid_ratio_DO_NOT_USE COMMENT
'The original complaints_per_100_restaurants ratio. Divides a residential numerator (70% of complaints) by a commercial denominator (restaurant count), which are uncorrelated (r = -0.03). Confirmed to invert rankings: see Block Three notebook queries 3.1-3.4. Retained only as an audit trail — never display or rank on this column.';